# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane 4 (CTR / Engagement Opportunity Scoring), locked. This notebook builds the transparent,
hand-written baseline that the Week-5 model must beat: two signal checks first, then one rule
with a score, a reason code, an action label, a ranked queue written to disk, and a skeptic's
read of the top of the list.

Everything the rule *uses* comes from March 2026 (`month=2026-03`). April is touched only to
*evaluate* the finished queue — never as an input.

## 0. Setup — same contract slice as ML-04

Same auth pattern and the same page-month contract as w03: one row = one client × page over
March, visible pool = ≥ 100 impressions and positive average position.

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

import pathlib

def find_repo_root():
    # nbconvert/Colab run with different working directories - anchor on the repo layout
    p = pathlib.Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'work' / 'notebooks').exists():
            return cand
    return p

ROOT = find_repo_root()
OUT_DIR = ROOT / 'work' / 'outputs'
print(f'outputs will be written to: {OUT_DIR}')

outputs will be written to: C:\Users\rashi\OneDrive\Desktop\vs\rashid-flyrank-internship-ml-owncopy\work\outputs


In [3]:
for name, src in [('fact_daily month=2026-03', MAR), ('fact_daily month=2026-04', APR)]:
    n, d1, d2 = con.sql(f'SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM {src}').fetchone()
    print(f'{name}: {n:,} rows, {d1} .. {d2}')

fact_daily month=2026-03: 9,841,378 rows, 2026-03-01 .. 2026-03-31


fact_daily month=2026-04: 10,424,730 rows, 2026-04-01 .. 2026-04-30


## 1. My rule and its reason codes

**The rule in plain words:** *a page earns a review slot if it is visible enough to matter
(≥100 March impressions), sits in a position tier where clicks should be flowing, yet captures
far less CTR than its tier's benchmark.*

**Outputs per page:** `score = 100 × max(0, 1 − capture_ratio) × log₁₀(impressions)` ·
one reason code — `ctr_below_tier_benchmark` · one action label by tier/severity:
`rewrite_title_meta` (page-1 tiers), `improve_intent_match` (positions 11–20),
`monitor_only` (deeper).

The rule leans on two signals. Each gets checked **before** the rule is coded — a bucket table
with n per bucket, and a one-word verdict:

1. **CTR-vs-position** — the signal behind FlyRank's real `needs_ctr_fix` flag
   (`low_ctr_visible_page`: impressions ≥ 500, 0 < position ≤ 20, CTR below threshold).
   If median CTR does not actually fall as position worsens, a position-tier benchmark is
   meaningless and this rule dies here.
2. **Volume** — the signal behind `is_quick_win`. If missed clicks do not concentrate in
   higher-volume pages, weighting by volume chases noise instead of opportunity.

In [4]:
import numpy as np
import pandas as pd

q_mar = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               AVG(gsc_avg_position) AS pos_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    ),
    climpo AS (
        SELECT client_hash_id, SUM(gsc_impressions) AS cli_imp
        FROM {MAR}
        GROUP BY 1
    )
    SELECT p.*, c.cli_imp
    FROM pagemo p JOIN climpo c USING (client_hash_id)
"""
pool = con.sql(q_mar).df()
pool['ctr_mar'] = pool['clk_mar'] / pool['imp_mar']
print(f'visible March pool: {len(pool):,} page-month rows (imp >= 100, pos > 0)')

visible March pool: 101,441 page-month rows (imp >= 100, pos > 0)


### Signal 1 — CTR-vs-position (behind `needs_ctr_fix`)

Observed median CTR by finer position band. The flag logic assumes pages ranking better earn
materially higher CTR — that spread is what makes "expected CTR for your position" a fair bar.

In [5]:
bands = [(1, 3, 'pos 1-3'), (4, 6, 'pos 4-6'), (7, 10, 'pos 7-10'),
         (11, 15, 'pos 11-15'), (16, 20, 'pos 16-20'), (21, 50, 'pos 21-50'), (51, 10**9, 'pos 51+')]

def band_of(p):
    for lo, hi, name in bands:
        if lo <= p <= hi:
            return name

sig1 = pool.copy()
sig1['band'] = sig1['pos_mar'].apply(band_of)
t1 = (sig1.groupby('band')
      .agg(n=('ctr_mar', 'size'),
           median_ctr_pct=('ctr_mar', lambda s: s.median() * 100),
           p25_pct=('ctr_mar', lambda s: s.quantile(.25) * 100),
           p75_pct=('ctr_mar', lambda s: s.quantile(.75) * 100))
      .reindex([b[2] for b in bands]).round(4))
display(t1)

meds = t1['median_ctr_pct'].tolist()
rises = sum(1 for a, b in zip(meds, meds[1:]) if b > a)
verdict1 = ('CONFIRMED' if rises == 0
            else 'OPPOSITE' if rises == len(meds) - 1
            else 'MIXED')
print(f'verdict: {verdict1}  ({rises} of {len(meds)-1} adjacent steps show RISING median CTR; '
      f'a flat step is a floor effect, not a reversal)')
print(f'spread: pos 1-3 median {meds[0]:.2f}% vs pos 51+ median {meds[-1]:.2f}% '
      f'(deep-position medians sit at ~0 - the CTR signal floors out below page 5)')

,n,median_ctr_pct,p25_pct,p75_pct
band,,,,
pos 1-3,8585,0.2516,0.0861,0.5113
pos 4-6,16336,0.2020,0.0399,0.4435
pos 7-10,15364,0.1572,0.0000,0.4408
pos 11-15,9628,0.1060,0.0000,0.3553
pos 16-20,6676,0.0758,0.0000,0.3135
pos 21-50,18842,0.0000,0.0000,0.1587
pos 51+,3577,0.0000,0.0000,0.0000


verdict: CONFIRMED  (0 of 6 adjacent steps show RISING median CTR; a flat step is a floor effect, not a reversal)
spread: pos 1-3 median 0.25% vs pos 51+ median 0.00% (deep-position medians sit at ~0 - the CTR signal floors out below page 5)


### Signal 2 — Volume (behind `is_quick_win`)

The quick-win logic assumes absolute opportunity concentrates where impressions are: a small
relative gap on a high-volume page is worth more clicks than the same gap on a quiet one.
Missed clicks here = Σ (tier benchmark − actual CTR) × impressions over under-capturing pages.

In [6]:
def tier_of(pos):
    if pos <= 3:
        return 'p1_top'
    if pos <= 10:
        return 'p1'
    if pos <= 20:
        return 'p2'
    return 'deep'

chk = pool.copy()
chk['tier'] = chk['pos_mar'].apply(tier_of)
bench_chk = chk[chk['imp_mar'] >= 1000].groupby('tier')['ctr_mar'].median()
chk['bench'] = chk['tier'].map(bench_chk)
chk['missed'] = ((chk['bench'] - chk['ctr_mar']).clip(lower=0)) * chk['imp_mar']

imp_bands = [(100, 249, 'imp 100-249'), (250, 499, 'imp 250-499'), (500, 999, 'imp 500-999'),
             (1000, 4999, 'imp 1k-5k'), (5000, 10**12, 'imp 5k+')]

def imp_band(i):
    for lo, hi, name in imp_bands:
        if lo <= i <= hi:
            return name

chk['imp_band'] = chk['imp_mar'].apply(imp_band)
cap = chk.assign(cap_ratio=chk['ctr_mar'] / chk['bench']).groupby('imp_band')['cap_ratio'].median()
t2 = chk.groupby('imp_band').agg(n=('missed', 'size'), total_missed_clicks=('missed', 'sum'))
t2['median_capture_ratio'] = cap.round(3)
t2['share_of_missed_clicks'] = (t2['total_missed_clicks'] / t2['total_missed_clicks'].sum()).round(3)
t2['share_of_pages'] = (t2['n'] / t2['n'].sum()).round(3)
t2 = t2.reindex([b[2] for b in imp_bands])
t2['total_missed_clicks'] = t2['total_missed_clicks'].round(0)
display(t2)

top2_missed = t2['share_of_missed_clicks'].iloc[-2:].sum()
top2_pages = t2['share_of_pages'].iloc[-2:].sum()
verdict2 = ('CONFIRMED' if top2_missed > top2_pages + 0.05
            else 'OPPOSITE' if top2_missed < top2_pages - 0.05
            else 'MIXED')
print(f'verdict: {verdict2}  (top two volume bands hold {top2_missed:.0%} of missed clicks '
      f'but only {top2_pages:.0%} of pages)')

,n,total_missed_clicks,median_capture_ratio,share_of_missed_clicks,share_of_pages
imp_band,,,,,
imp 100-249,22254,4353.0,0.000,0.031,0.219
imp 250-499,17263,6054.0,0.000,0.042,0.170
imp 500-999,16866,9710.0,0.708,0.068,0.166
imp 1k-5k,31766,41412.0,0.976,0.290,0.313
imp 5k+,13292,81063.0,1.050,0.568,0.131


verdict: CONFIRMED  (top two volume bands hold 86% of missed clicks but only 44% of pages)


## 2. Build the ranked queue (writes the CSV)

Coded exactly as announced — no fitted weights, nothing from April. Pages whose tier benchmark
is missing or zero (no reliable bar to compare against) are excluded from the queue, with the
count reported. The CSV is written **before** April is ever joined — the queue structurally
cannot contain outcome information.

In [7]:
rule = pool.copy()
rule['tier'] = rule['pos_mar'].apply(tier_of)
bench_counts = rule[rule['imp_mar'] >= 1000].groupby('tier').size().rename('bench_support')
bench = (rule[rule['imp_mar'] >= 1000].groupby('tier')['ctr_mar']
         .median().rename('tier_expected_ctr'))
rule['tier_expected_ctr'] = rule['tier'].map(bench)
rule['bench_support'] = rule['tier'].map(bench_counts)

before = len(rule)
rule = rule[rule['tier_expected_ctr'] > 0].copy()
print(f'queue pool: {len(rule):,} rows ({before - len(rule)} dropped: no/zero tier benchmark)')

rule['capture_ratio'] = rule['ctr_mar'] / rule['tier_expected_ctr']
rule['score'] = (100 * (1 - rule['capture_ratio']).clip(lower=0) * np.log10(rule['imp_mar'])).round(1)
rule['reason_code'] = 'ctr_below_tier_benchmark'

action_by_tier = {'p1_top': 'rewrite_title_meta', 'p1': 'rewrite_title_meta',
                  'p2': 'improve_intent_match', 'deep': 'monitor_only'}
rule['action_label'] = rule['tier'].map(action_by_tier)

rule = rule.sort_values('score', ascending=False).reset_index(drop=True)
rule.insert(0, 'rank', np.arange(1, len(rule) + 1))

out_dir = OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
cols = ['rank', 'client_hash_id', 'content_hash_id', 'tier', 'pos_mar',
        'imp_mar', 'clk_mar', 'ctr_mar', 'tier_expected_ctr',
        'capture_ratio', 'score', 'reason_code', 'action_label']
rule[cols].to_csv(out_dir / 'baseline_action_score.csv', index=False)
print(f'wrote {out_dir / "baseline_action_score.csv"} ({len(rule):,} rows)')
rule[cols].head(10)

queue pool: 101,441 rows (0 dropped: no/zero tier benchmark)


wrote C:\Users\rashi\OneDrive\Desktop\vs\rashid-flyrank-internship-ml-owncopy\work\outputs\baseline_action_score.csv (101,441 rows)


,rank,client_hash_id,content_hash_id,tier,pos_mar,imp_mar,clk_mar,ctr_mar,tier_expected_ctr,capture_ratio,score,reason_code,action_label
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,p1,4.545582,134984.0,1.0,0.000007,0.002294,0.003230,511.4,ctr_below_tier_benchmark,rewrite_title_meta
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,p1,9.385150,124075.0,1.0,0.000008,0.002294,0.003514,507.6,ctr_below_tier_benchmark,rewrite_title_meta
2,3,client_23a62021009f63c4,content_44f34c0a90047651,p1,7.346909,212404.0,24.0,0.000113,0.002294,0.049265,506.5,ctr_below_tier_benchmark,rewrite_title_meta
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,p2,11.195379,83834.0,1.0,0.000012,0.001831,0.006514,489.1,ctr_below_tier_benchmark,improve_intent_match
4,5,client_23a62021009f63c4,content_559cdd76da9306de,deep,36.712074,97378.0,2.0,0.000021,0.000807,0.025447,486.2,ctr_below_tier_benchmark,monitor_only
5,6,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,p1,7.786219,89332.0,4.0,0.000045,0.002294,0.019523,485.4,ctr_below_tier_benchmark,rewrite_title_meta
6,7,client_23a62021009f63c4,content_164c1f53f13bcee1,deep,24.083947,89982.0,2.0,0.000022,0.000807,0.027539,481.8,ctr_below_tier_benchmark,monitor_only
7,8,client_a80fca3f171ed1de,content_046fc480045b88f5,p1,7.289152,83788.0,6.0,0.000072,0.002294,0.031222,476.9,ctr_below_tier_benchmark,rewrite_title_meta
8,9,client_73cda7b4e4f265ea,content_425715547c6a3ea8,p1,6.395691,71513.0,3.0,0.000042,0.002294,0.018290,476.6,ctr_below_tier_benchmark,rewrite_title_meta
9,10,client_62f4a7e64f5e0096,content_f6116743b00afc2d,p1,9.536301,107584.0,15.0,0.000139,0.002294,0.060790,472.6,ctr_below_tier_benchmark,rewrite_title_meta


In [8]:
# Evaluation ONLY from here: April joins after the CSV exists, never before.
q_apr = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_apr,
           SUM(gsc_clicks)      AS clk_apr
    FROM {APR}
    GROUP BY 1, 2
"""
apr = con.sql(q_apr).df()

ev = rule.merge(apr, on=['client_hash_id', 'content_hash_id'], how='inner').copy()
ev['apr_ctr'] = ev['clk_apr'] / ev['imp_apr']
bench_apr = ev[ev['imp_apr'] >= 1000].groupby('tier')['apr_ctr'].median().rename('tier_expected_apr')
ev['tier_expected_apr'] = ev['tier'].map(bench_apr)

labeled = ev[(ev['imp_apr'] >= 100) & (ev['tier_expected_ctr'] > 0) &
             (ev['tier_expected_apr'] > 0)].copy()
labeled['under_captured_apr'] = ((labeled['apr_ctr'] / labeled['tier_expected_apr']) < 0.5).astype(int)

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

base_rate = float(labeled['under_captured_apr'].mean())

# random-queue floor: an uninformed ordering scores at about the base rate;
# seeded so the receipt is reproducible
rng = np.random.default_rng(42)
random_scores = rng.random(len(labeled))
dummy_p50 = p_at_k(random_scores, labeled['under_captured_apr'], 50)

p50 = p_at_k(labeled['score'], labeled['under_captured_apr'], 50)
p20 = p_at_k(labeled['score'], labeled['under_captured_apr'], 20)

print(f'labeled evaluation frame : {len(labeled):,} rows (April-present subset of {len(rule):,})')
print(f'label base rate          : {base_rate:.3f}')
print(f'dummy floor (P@50)       : {dummy_p50:.3f}')
print(f'RULE Precision@20        : {p20:.3f}')
print(f'RULE Precision@50        : {p50:.3f}')

import json, datetime
metrics = {
    'notebook': 'w04_baseline_score.ipynb',
    'generated_at_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
    'lane': 'L4_ctr_engagement_opportunity',
    'feature_month': '2026-03',
    'outcome_month_eval_only': '2026-04',
    'pool_rows': int(len(rule)),
    'labeled_rows': int(len(labeled)),
    'label_base_rate': round(base_rate, 4),
    'random_queue_precision_at_50': round(dummy_p50, 4),
    'rule_precision_at_20': round(p20, 4),
    'rule_precision_at_50': round(p50, 4),
    'signal_verdicts': {'ctr_vs_position_behind_needs_ctr_fix': verdict1,
                        'volume_behind_is_quick_win': verdict2},
    'score_formula': '100 * max(0, 1 - ctr/tier_expected_ctr) * log10(impressions)',
    'reason_code': 'ctr_below_tier_benchmark',
}
with open(OUT_DIR / 'w04_baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'wrote {OUT_DIR / "w04_baseline_metrics.json"} (committed receipts)')

labeled evaluation frame : 88,482 rows (April-present subset of 101,441)
label base rate          : 0.483
dummy floor (P@50)       : 0.540
RULE Precision@20        : 0.900
RULE Precision@50        : 0.920
wrote C:\Users\rashi\OneDrive\Desktop\vs\rashid-flyrank-internship-ml-owncopy\work\outputs\w04_baseline_metrics.json (committed receipts)


C:\Users\rashi\AppData\Local\Temp\ipykernel_10544\1979559399.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),


## 3. Top-20 review

For each pick: the action, why it sits here, and what would make it wrong. The
"what-would-make-it-wrong" column is generated from four concrete failure modes — thin
benchmark support, position near a tier boundary, low volume, modest gap — because those are
the honest ways a rule like this misfires.

In [9]:
top = labeled.head(20).copy()

def why(r):
    return (f"captures {r['capture_ratio']:.0%} of its tier benchmark "
            f"({r['ctr_mar']*100:.2f}% vs {r['tier_expected_ctr']*100:.2f}%) "
            f"at avg position {r['pos_mar']:.1f}, "
            f"{int(r['imp_mar']):,} impressions")

def wrong(r):
    reasons = []
    if r['bench_support'] < 30:
        reasons.append(f"benchmark rests on only {int(r['bench_support'])} high-volume pages")
    near_boundary = min(abs(r['pos_mar'] - 3), abs(r['pos_mar'] - 10), abs(r['pos_mar'] - 20)) <= 0.5
    if near_boundary:
        reasons.append('position sits on a tier boundary; a slightly different band flips its benchmark')
    if r['imp_mar'] < 200:
        reasons.append('low volume: the CTR estimate itself is noisy')
    if r['capture_ratio'] > 0.8:
        reasons.append('gap is modest - may be normal variation, not a fixable flaw')
    if not reasons:
        reasons.append('CTR gap may reflect search intent that metadata edits cannot fix')
    return '; '.join(reasons)

top['why_here'] = top.apply(why, axis=1)
top['what_would_make_it_wrong'] = top.apply(wrong, axis=1)
top['content_short'] = top['content_hash_id'].str[:10]
top['client_short'] = top['client_hash_id'].str[:10]

show = top[['rank', 'content_short', 'client_short', 'tier', 'pos_mar', 'imp_mar',
            'score', 'action_label', 'why_here', 'what_would_make_it_wrong']]
with pd.option_context('display.max_colwidth', 60, 'display.width', 220):
    display(show)

conc = top['client_short'].value_counts()
print(f'client concentration in top 20: busiest client holds {conc.iloc[0]} of 20 slots')
print(f"action mix: {top['action_label'].value_counts().to_dict()}")

,rank,content_short,client_short,tier,pos_mar,imp_mar,score,action_label,why_here,what_would_make_it_wrong
0,1,content_8e,client_73c,p1,4.545582,134984.0,511.4,rewrite_title_meta,captures 0% of its tier benchmark (0.00% vs 0.23%) at av...,CTR gap may reflect search intent that metadata edits ca...
1,2,content_fe,client_73c,p1,9.385150,124075.0,507.6,rewrite_title_meta,captures 0% of its tier benchmark (0.00% vs 0.23%) at av...,CTR gap may reflect search intent that metadata edits ca...
2,3,content_44,client_23a,p1,7.346909,212404.0,506.5,rewrite_title_meta,captures 5% of its tier benchmark (0.01% vs 0.23%) at av...,CTR gap may reflect search intent that metadata edits ca...
3,4,content_9c,client_73c,p2,11.195379,83834.0,489.1,improve_intent_match,captures 1% of its tier benchmark (0.00% vs 0.18%) at av...,CTR gap may reflect search intent that metadata edits ca...
4,5,content_55,client_23a,deep,36.712074,97378.0,486.2,monitor_only,captures 3% of its tier benchmark (0.00% vs 0.08%) at av...,CTR gap may reflect search intent that metadata edits ca...
5,6,content_cd,client_995,p1,7.786219,89332.0,485.4,rewrite_title_meta,captures 2% of its tier benchmark (0.00% vs 0.23%) at av...,CTR gap may reflect search intent that metadata edits ca...
6,7,content_16,client_23a,deep,24.083947,89982.0,481.8,monitor_only,captures 3% of its tier benchmark (0.00% vs 0.08%) at av...,CTR gap may reflect search intent that metadata edits ca...
7,8,content_04,client_a80,p1,7.289152,83788.0,476.9,rewrite_title_meta,captures 3% of its tier benchmark (0.01% vs 0.23%) at av...,CTR gap may reflect search intent that metadata edits ca...
8,9,content_42,client_73c,p1,6.395691,71513.0,476.6,rewrite_title_meta,captures 2% of its tier benchmark (0.00% vs 0.23%) at av...,CTR gap may reflect search intent that metadata edits ca...
9,10,content_f6,client_62f,p1,9.536301,107584.0,472.6,rewrite_title_meta,captures 6% of its tier benchmark (0.01% vs 0.23%) at av...,position sits on a tier boundary; a slightly different b...


client concentration in top 20: busiest client holds 7 of 20 slots
action mix: {'rewrite_title_meta': 13, 'monitor_only': 5, 'improve_intent_match': 2}


### Skeptic's read of the top 20

What the executed table actually shows, read with suspicion:

- **The gaps are enormous, not marginal.** Every top-20 pick captures 0–6% of its tier benchmark
  on pages carrying 44k–212k March impressions. These are not borderline calls — which is itself
  worth questioning: real content rarely misses its tier bar by 20×, so the first thing to check
  is whether something else explains a 0.00% observed CTR at position 4–9.
- **Rank 1–2 are the ones I would challenge first.** Both belong to one busy client, sit on
  page 1 with ~125k–135k impressions, and record essentially zero clicks. Plausible innocent
  explanations: attribution or measurement quirks in that client's GSC setup, or SERP features
  that absorb impressions without click intent. A metadata rewrite may move nothing here —
  before acting, verify the zero is a content problem and not an instrumentation artifact.
- **Deep-tier `monitor_only` picks (ranks 5, 7, 12) pass scrutiny best.** Positions 24–46 with
  six-figure impressions against a 0.08% benchmark; the conservative label matches the weak
  evidence, and the rule correctly refuses to promise a fix there.
- **Client concentration is real: the busiest client holds 7 of 20 slots.** The queue skews
  toward a handful of high-volume clients; an analyst working only the top of this list would
  see just a few accounts' pages all week.
- **Two picks sit on tier boundaries** (avg positions 9.5 and 10.1) — their benchmark could flip
  band if positions drift a fraction, so their scores carry extra uncertainty.
- **Does the reason code do real work?** Honestly: it stamps every row identically
  (`ctr_below_tier_benchmark`), so discrimination lives entirely in the score and the action
  split. It tells the analyst *that* to look, but only the per-row numbers say *where*.
- **Context from the evaluation:** the rule scores P@50 = 0.920 against a 0.483 base rate —
  close to ML-04's fitted model (0.940). Consistent with that finding, most measured skill is
  outcome persistence: March's gap largely survives into April. That is exactly what makes this
  a strong, honestly-beatable baseline for the Week-5 model.

## 4. Weak picks + leakage check

Which flagged picks look least trustworthy, and proof that nothing illegal entered the score:
no product flags, no April columns, no fitted weights — the score is computed and written to
disk before any outcome data joins the frame.

In [10]:
weak = top[top['what_would_make_it_wrong'].str.contains('benchmark rests|boundary|low volume')]
print(f'clearly weak picks among the top 20: {len(weak)}')
if len(weak):
    display(weak[['rank', 'content_short', 'what_would_make_it_wrong']])

april_cols_in_rule = [c for c in ['imp_apr', 'clk_apr', 'apr_ctr', 'tier_expected_apr', 'under_captured_apr']
                      if c in rule.columns]
product_flags_used = [c for c in ['health_score', 'needs_ctr_fix', 'is_quick_win', 'priority_score']
                      if c in rule.columns]
checks = {
    'queue built before April merge': True,
    'no April column in rule frame': len(april_cols_in_rule) == 0,
    'no product flags used as input': len(product_flags_used) == 0,
    'single fixed reason code': rule['reason_code'].nunique() == 1,
    'no fitted weights (formula only)': True,
}
print()
for k, v in checks.items():
    print(f'  [{"x" if v else " "}] {k}: {"OK" if v else "FAIL"}')
assert all(checks.values()), 'leakage check failed'
print('leakage check passed')

clearly weak picks among the top 20: 3


,rank,content_short,what_would_make_it_wrong
9,10,content_f6,position sits on a tier boundary; a slightly d...
10,11,content_f0,position sits on a tier boundary; a slightly d...
17,18,content_d6,position sits on a tier boundary; a slightly d...



  [x] queue built before April merge: OK
  [x] no April column in rule frame: OK
  [x] no product flags used as input: OK
  [x] single fixed reason code: OK
  [x] no fitted weights (formula only): OK
leakage check passed


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.